In [ ]:
import pandas as pd
import torch
from bert_score import score
from tqdm import tqdm

# 1. 평가할 데이터 로드 (이전 단계에서 생성한 파일)
df_eval = pd.read_csv("./output/final_generation_results.csv")

# 데이터 전처리 (결측치 제거 및 리스트 변환)
# refs: 정답(Corrected), cands: 모델이 생성한 문장(Final Output)
# 주의: 학습 시 사용했던 'final_corrected' 컬럼이 정답지 역할을 합니다.
# 만약 Testset에 'final_corrected'가 있다면 그것을 사용하세요.
refs = df_eval['final_corrected'].tolist()  
cands = df_eval['final_output'].tolist()

# ---------------------------------------------------------
# Metric 1: BERTScore (한국어 특화)
# ---------------------------------------------------------
print("📊 Calculating BERTScore (Model: klue/roberta-large)...")

# device 설정
device = "cuda" if torch.cuda.is_available() else "cpu"

P, R, F1 = score(
    cands, 
    refs, 
    model_type="klue/roberta-large",  # [핵심] 한국어 모델 지정
    lang="ko", 
    verbose=True,
    device=device,
    batch_size=32
)

# 결과 저장
df_eval['bert_f1'] = F1.numpy()

print(f"✅ BERTScore 평균 F1: {df_eval['bert_f1'].mean():.4f}")

# ---------------------------------------------------------
# Metric 2: LLM-as-a-Judge (Quality Check)
# ---------------------------------------------------------
# 작성자님의 기존 GeminiProcessor 클래스를 활용한다고 가정합니다.
# 평가 기준: 의도 반영(Intent), 유창성(Fluency), 의미 보존(Meaning)

def build_eval_prompt(batch_data):
    input_text = ""
    for item in batch_data:
        input_text += f"""
        [Case ID: {item.get('index', 0)}]
        - Original: "{item['input_sentence']}"
        - User Intent: {item['tag_intensity']} / {item['tag_field']}
        - Generated Correction: "{item['final_output']}"
        - Ground Truth: "{item['final_corrected']}"
        -------------------------
        """
    
    return f"""
    Role: Senior Korean Editor.
    Task: Evaluate the quality of the 'Generated Correction' compared to the 'Ground Truth' and 'User Intent'.
    
    [Scoring Rubric (1-5)]
    5: Perfect. Captures intent perfectly, natural Korean flow.
    4: Good. Minor awkwardness but intent is clear.
    3: Fair. Meaning is preserved but intent (style) is weak.
    2: Poor. Grammatical errors or meaning changed.
    1: Bad. Hallucination or failed correction.

    [Input Data]
    {input_text}

    [Output Requirement]
    Return JSON list: [{{ "id": case_id, "quality_score": int, "reason": "short comment" }}]
    """

# (여기서 GeminiProcessor로 위 프롬프트를 실행하여 'quality_score'를 얻는 로직 수행)
# ...

# ---------------------------------------------------------
# 결과 분석 및 저장
# ---------------------------------------------------------
# Intensity별 성능 분석
print("\n📈 [Intensity별 성능 분석]")
print(df_eval.groupby('tag_intensity')['bert_f1'].mean())

# Field별 성능 분석
print("\n📈 [Field별 성능 분석]")
print(df_eval.groupby('tag_field')['bert_f1'].mean())

df_eval.to_csv("./output/final_metric_evaluation.csv", index=False, encoding='utf-8-sig')
print("💾 평가 완료 및 저장: ./output/final_metric_evaluation.csv")